<a href="https://colab.research.google.com/github/BrajanNieto/DeepLearning/blob/main/Grupo4_Laboratorio03_XTTSv2%20_CoquiAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Deep Learning - PatchTST from Scratch**

---

* This repository contains the implementation of the PatchTST (Patch Time Series Transformer) architecture from scratch. It focuses on multivariate time series forecasting by applying patching techniques and transformer-based models to efficiently capture long-term dependencies.

**Authors:**

Lopez Medina, Sebastian  
[sebastian.lopez@utec.edu.pe](mailto:sebastian.lopez@utec.edu.pe)

Nieto Espinoza, Brajan  
[brajan.nieto@utec.edu.pe](mailto:brajan.nieto@utec.edu.pe)

Tapia Chasquibol, Mateo  
[mateo.tapia@utec.edu.pe](mailto:mateo.tapia@utec.edu.pe)

---

# 0. Configuración de Entorno, Librerías y Dataser

* **Librerías:** Importación de PyTorch, herramientas de manejo de datos y visualización.
* **Reproducibilidad:** Configuración de `SEED` y detección de aceleración por hardware (CUDA).
* **Fuentes de Datos:** Definición de diccionarios de URLs y constantes para la selección de variables del dataset.

In [11]:
!pip install -q coqui-tts torch torchaudio
!apt-get -qq install ffmpeg

In [ ]:
import requests
import os


# 1. Automatización de Ingesta: Sincronización de Activos desde GitHub

Implementación de un flujo de trabajo para la **descarga automatizada de recursos de audio** desde un repositorio remoto de GitHub hacia el entorno local (Google Colab). Este componente actúa como el puente de datos inicial, asegurando que el modelo tenga acceso a las muestras de voz necesarias para el procesamiento posterior sin necesidad de cargas manuales.

* **Configuración de Endpoints de API:** Define dinámicamente las rutas de acceso utilizando la API de GitHub para listar contenidos y la URL de archivos *raw* para la descarga. Esta estructura permite una integración flexible con diferentes ramas (`main`) y carpetas específicas (`voices`), facilitando la escalabilidad del dataset.
* **Filtrado Selectivo de Formatos:** Implementa una validación basada en extensiones (`.ogg`, `.wav`, `.mp3`, etc.) para asegurar que solo se procesen archivos de audio compatibles. Esto optimiza el consumo de ancho de banda y evita errores en las etapas de pre-procesamiento de señales.
* **Gestión de Persistencia Local:** Automatiza la creación de directorios en el entorno de ejecución (`/content/audios_originales`) mediante `os.makedirs`. El código garantiza que la infraestructura de carpetas sea consistente antes de iniciar la escritura de los binarios descargados.
* **Mapeo de Metadatos y Diccionario de Referencia:** Construye un diccionario (`audios_descargados`) que vincula el nombre de la "persona" (extraído del nombre del archivo) con su ruta local absoluta. Este mapeo es crítico para la trazabilidad y el etiquetado de datos durante el entrenamiento o la inferencia del modelo.
* **Monitoreo de Integridad de Transferencia:** Incorpora un sistema de logging en tiempo real que reporta el tamaño de los archivos (en KB) y el estado de la descarga. El uso de `raise_for_status()` asegura que el proceso se detenga inmediatamente ante cualquier error de red (HTTP 4xx o 5xx), manteniendo la robustez del pipeline.

In [20]:
# ============================================================
# CONFIGURACIÓN - Tu repositorio
# ============================================================
GITHUB_USER = 'BrajanNieto'
GITHUB_REPO = 'DeepLearning'
GITHUB_BRANCH = 'main'
VOICES_FOLDER = 'voices'

# URLs
API_URL = f'https://api.github.com/repos/{GITHUB_USER}/{GITHUB_REPO}/contents/{VOICES_FOLDER}?ref={GITHUB_BRANCH}'
RAW_BASE = f'https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}/{VOICES_FOLDER}'

# Directorio local
LOCAL_DIR = '/content/audios_originales'
os.makedirs(LOCAL_DIR, exist_ok=True)

# Listar archivos en la carpeta voices/
print(f'Consultando: {API_URL}')
response = requests.get(API_URL)
response.raise_for_status()
files = response.json()

# Filtrar solo archivos de audio
AUDIO_EXTENSIONS = ('.ogg', '.wav', '.mp3', '.m4a', '.flac')
audio_files = [f for f in files if f['name'].lower().endswith(AUDIO_EXTENSIONS)]

print(f'\n Archivos de audio encontrados: {len(audio_files)}')
for f in audio_files:
    print(f' {f["name"]} ({f["size"]/1024:.1f} KB)')

# Descargar cada archivo
audios_descargados = {}

for f in audio_files:
    nombre = f['name']
    # El nombre de la persona es el nombre del archivo sin extensión
    persona = os.path.splitext(nombre)[0]
    url = f'{RAW_BASE}/{nombre}'
    local_path = os.path.join(LOCAL_DIR, nombre)

    print(f'\n Descargando: {nombre}...')
    r = requests.get(url)
    r.raise_for_status()
    with open(local_path, 'wb') as out:
        out.write(r.content)

    audios_descargados[persona] = local_path
    print(f' {persona} → {local_path} ({len(r.content)/1024:.1f} KB)')

print(f'\n{"="*50}')
print(f' {len(audios_descargados)} audios descargados')
print(f'{"="*50}')
for persona, path in audios_descargados.items():
    print(f' {persona}: {path}')

Consultando: https://api.github.com/repos/BrajanNieto/DeepLearning/contents/voices?ref=main

 Archivos de audio encontrados: 3
 BNE_Voice.ogg (60.8 KB)
 MTC_Voice.ogg (70.5 KB)
 SLM_Voice.ogg (57.8 KB)

 Descargando: BNE_Voice.ogg...
 BNE_Voice → /content/audios_originales/BNE_Voice.ogg (60.8 KB)

 Descargando: MTC_Voice.ogg...
 MTC_Voice → /content/audios_originales/MTC_Voice.ogg (70.5 KB)

 Descargando: SLM_Voice.ogg...
 SLM_Voice → /content/audios_originales/SLM_Voice.ogg (57.8 KB)

 3 audios descargados
 BNE_Voice: /content/audios_originales/BNE_Voice.ogg
 MTC_Voice: /content/audios_originales/MTC_Voice.ogg
 SLM_Voice: /content/audios_originales/SLM_Voice.ogg


# 2. Normalización de Señal y Pre-procesamiento de Audio

Implementación del pipeline de **estandarización acústica** utilizando `FFmpeg` y `Torchaudio`. Esta fase es crítica para garantizar que todos los activos de audio, independientemente de su origen o formato inicial, posean las mismas propiedades físicas antes de ser consumidos por el modelo de aprendizaje profundo.

* **Transcodificación y Resampling:** Ejecuta un subproceso de `ffmpeg` para convertir los archivos a formato **WAV** con una frecuencia de muestreo de **22,050 Hz**. Este valor es el estándar para modelos de síntesis de voz (TTS) y clonación, ya que equilibra la fidelidad del audio con la eficiencia computacional.
* **Estandarización de Canales y Bits:** Fuerza la conversión a **monoaural (1 canal)** y una profundidad de bits de **16-bit (s16)**. Eliminar la información estéreo y normalizar el formato de muestra reduce la varianza en los datos de entrada, permitiendo que el modelo se enfoque puramente en las características del timbre de voz.
* **Validación de Atributos mediante Torchaudio:** Carga la forma de onda resultante para verificar la integridad del archivo y extraer metadatos técnicos. Este paso asegura que el tensor de audio (`waveform`) se haya generado correctamente y sea compatible con los tensores de PyTorch.
* **Control de Calidad por Duración Temporal:** Implementa un filtro de umbral para evaluar la longitud del audio. Dado que el entrenamiento requiere suficiente contexto fonético, el sistema emite alertas para audios menores a **6 segundos**, asegurando que el dataset mantenga un estándar de calidad óptimo (idealmente entre 10 y 30 segundos).
* **Gestión de Errores de Subproceso:** Captura el `returncode` de la ejecución de sistema para identificar fallos en la decodificación. Al reportar los últimos caracteres del `stderr`, permite un diagnóstico rápido de archivos corruptos o codecs no soportados sin detener el flujo de trabajo principal.

In [21]:
import subprocess

audios_procesados = {}

for persona, filepath in audios_descargados.items():
    output_path = os.path.join(LOCAL_DIR, f'{persona}_procesado.wav')

    # ffmpeg: cualquier formato → WAV 22050Hz mono 16bit
    cmd = [
        'ffmpeg', '-y', '-i', filepath,
        '-ar', '22050',
        '-ac', '1',
        '-sample_fmt', 's16',
        output_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f'Error procesando {persona}: {result.stderr[-200:]}')
        continue

    # Verificar duración
    import torchaudio
    waveform, sr = torchaudio.load(output_path)
    duracion = waveform.shape[1] / sr

    audios_procesados[persona] = output_path
    print(f' {persona}: {duracion:.1f}s | {sr}Hz | mono')

    if duracion < 6:
        print(f' Audio muy corto ({duracion:.1f}s). Mínimo recomendado: 6s, ideal: 10-30s')

 BNE_Voice: 29.0s | 22050Hz | mono
 MTC_Voice: 30.8s | 22050Hz | mono
 SLM_Voice: 24.9s | 22050Hz | mono


# 3. Inicialización del Ecosistema de Síntesis de Voz (XTTS v2)

Configuración y despliegue del motor de **Text-to-Speech (TTS)** basado en la arquitectura de vanguardia `XTTS v2`. Esta fase establece la infraestructura computacional necesaria para realizar clonación de voz multilingüe de alta fidelidad, gestionando la transferencia del modelo a hardware especializado (GPU) para optimizar los tiempos de inferencia.

* **Gestión Dinámica de Aceleración de Hardware:** Detecta automáticamente la disponibilidad de núcleos **CUDA**. Al priorizar la ejecución en GPU sobre CPU, se garantiza una generación de audio en tiempo real, reduciendo drásticamente la latencia en el procesamiento de tensores de gran escala.
* **Instanciación del Modelo Multilingüe:** Carga el checkpoint `xtts_v2` desde el repositorio oficial de *Coqui TTS*. Este modelo destaca por su capacidad de mantener la identidad vocal a través de diversos idiomas, utilizando una arquitectura de GPT condicionada por embeddings de voz.
* **Pipeline de Carga Diferida (Lazy Loading):** Implementa un proceso de descarga y deserialización de pesos que solo ocurre en la primera ejecución. Una vez en memoria, el modelo se mantiene listo para procesar múltiples solicitudes de síntesis sin recargas adicionales.
* **Exposición de la Arquitectura Interna:** Accede directamente al objeto `synthesizer.tts_model`. Esta técnica permite interactuar con las capas profundas del modelo (como el encoder de audio o el decoder de latentes) para tareas personalizadas de extracción de características o ajuste de parámetros finos que no están disponibles en la API de alto nivel.
* **Preparación del Entorno de Tensores:** Al ejecutar `tts.to(device)`, el sistema asegura que todos los pesos de la red neuronal y los buffers intermedios residan en el mismo espacio de memoria, evitando cuellos de botella por transferencia de datos entre el host (CPU) y el dispositivo (GPU).

In [22]:
import torch
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f' Dispositivo: {device}')

# Cargar XTTS v2
print('\n Descargando y cargando XTTS v2 (primera vez tarda ~2-3 min)...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2')
tts.to(device)
print(' Modelo XTTS v2 cargado')

# Acceder al modelo interno
modelo_xtts = tts.synthesizer.tts_model

 Dispositivo: cuda

 Descargando y cargando XTTS v2 (primera vez tarda ~2-3 min)...
 Modelo XTTS v2 cargado


# 4. Extracción de Latentes y Generación de Embeddings de Voz

Implementación del proceso de **codificación de identidad vocal**, donde se extraen las características biométricas del audio procesado para generar una representación matemática única (embedding). Este componente es el corazón de la clonación *zero-shot*, ya que permite al modelo XTTS v2 replicar una voz específica utilizando únicamente unos segundos de referencia.

* **Extracción de Conditioning Latents:** Utiliza el codificador interno de `XTTS v2` para obtener los `gpt_cond_latent` y el `speaker_embedding`. Estas representaciones capturan tanto la textura fonética fina como las características globales de la voz (tono, timbre y cadencia), permitiendo que el modelo condicione su generación de audio a la identidad del locutor.
* **Segmentación de Referencia (Chunking):** Configura parámetros críticos como `gpt_cond_len` y `gpt_cond_chunk_len` para fragmentar el audio de entrada en segmentos manejables. Este proceso de *chunking* asegura que la extracción de características sea consistente y que el modelo de atención pueda procesar la información temporal de la voz de manera eficiente.
* **Serialización de Pesos en Formato `.pth`:** Implementa la persistencia de datos utilizando `torch.save`. Al empaquetar los tensores junto con metadatos descriptivos (nombre de la persona y audio de origen), se crea un archivo de "huella digital" transportable que puede ser cargado instantáneamente en futuras sesiones de inferencia sin re-procesar el audio original.
* **Gestión de Memoria y Desplazamiento (CPU Offloading):** Aplica `.cpu()` a los tensores antes del guardado. Esta práctica es fundamental para asegurar que los archivos resultantes sean compatibles con sistemas que no posean GPU y para liberar memoria de video (VRAM) durante el ciclo de procesamiento masivo.
* **Validación de Dimensiones y Trazabilidad:** Monitorea las dimensiones de los tensores generados (shapes) y el tamaño del archivo en disco. El registro de estas dimensiones permite verificar que el proceso de codificación ha mantenido la densidad de información necesaria para una síntesis de alta fidelidad.

In [23]:
# Extraer y guardar embeddings
OUTPUT_DIR = '/content/voice_embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)

archivos_generados = {}

for persona, audio_path in audios_procesados.items():
    print(f'\n{"="*50}')
    print(f' Extrayendo embedding de: {persona}')
    print(f'{"="*50}')

    # Extraer conditioning latents
    gpt_cond_latent, speaker_embedding = modelo_xtts.get_conditioning_latents(
        audio_path=[audio_path],
        gpt_cond_len=30,
        gpt_cond_chunk_len=4,
        max_ref_length=30,
    )

    # Guardar como .pth
    save_path = os.path.join(OUTPUT_DIR, f'{persona}_voice.pth')
    torch.save({
        'gpt_cond_latent': gpt_cond_latent.cpu(),
        'speaker_embedding': speaker_embedding.cpu(),
        'nombre': persona,
        'source_audio': os.path.basename(audio_path),
    }, save_path)

    archivos_generados[persona] = save_path

    size_kb = os.path.getsize(save_path) / 1024
    print(f'   gpt_cond_latent:   {gpt_cond_latent.shape}')
    print(f'   speaker_embedding: {speaker_embedding.shape}')
    print(f'   Guardado: {save_path} ({size_kb:.1f} KB)')

print(f'\n{"="*50}')
print(f' ¡{len(archivos_generados)} embeddings extraídos!')
print(f'{"="*50}')


 Extrayendo embedding de: BNE_Voice
   gpt_cond_latent:   torch.Size([1, 32, 1024])
   speaker_embedding: torch.Size([1, 512, 1])
   Guardado: /content/voice_embeddings/BNE_Voice_voice.pth (132.0 KB)

 Extrayendo embedding de: MTC_Voice
   gpt_cond_latent:   torch.Size([1, 32, 1024])
   speaker_embedding: torch.Size([1, 512, 1])
   Guardado: /content/voice_embeddings/MTC_Voice_voice.pth (132.0 KB)

 Extrayendo embedding de: SLM_Voice
   gpt_cond_latent:   torch.Size([1, 32, 1024])
   speaker_embedding: torch.Size([1, 512, 1])
   Guardado: /content/voice_embeddings/SLM_Voice_voice.pth (132.0 KB)

 ¡3 embeddings extraídos!


# 5. Inferencia de Síntesis y Validación de Clonación (Zero-Shot)

Implementación del proceso de **síntesis de voz condicionada**, donde el modelo XTTS v2 utiliza los latentes previamente extraídos para generar nuevas secuencias de audio a partir de texto arbitrario. Esta fase final permite validar la fidelidad de la clonación y la capacidad del modelo para preservar la identidad vocal del locutor original en un entorno multilingüe.

* **Reconstitución de Contexto de Voz:** Recupera los tensores `gpt_cond_latent` y `speaker_embedding` desde los archivos `.pth` almacenados. El uso de `map_location=device` garantiza que los datos se carguen directamente en la memoria de la GPU, eliminando latencias de transferencia durante la ejecución del modelo.
* **Inferencia Condicionada por GPT:** Ejecuta el método `inference` del motor XTTS, integrando el texto de prueba con la identidad vocal cargada. El parámetro `language='es'` activa el soporte específico para fonemas en español, asegurando una prosodia y acentuación naturales acordes al idioma de destino.
* **Control de Variabilidad Estocástica:** Configura el parámetro `temperature=0.7` para balancear la creatividad y la estabilidad de la síntesis. Este valor permite que la generación tenga suficiente variación natural para evitar un tono robótico, sin comprometer la coherencia acústica de la voz clonada.
* **Renderizado y Post-procesamiento de Audio:** Convierte la salida del modelo (un vector de amplitudes) en un archivo **WAV** utilizando `torchaudio.save`. La tasa de muestreo se ajusta a **24,000 Hz**, optimizando la calidad de salida para la reproducción final y manteniendo la compatibilidad con reproductores estándar.
* **Visualización e Interfaz de Usuario:** Utiliza la librería `IPython.display` para integrar controles de reproducción de audio directamente en el entorno de trabajo. Este feedback inmediato es esencial para realizar pruebas de calidad (A/B testing) y ajustar el texto o los parámetros de inferencia en tiempo real.

In [24]:
from IPython.display import Audio, display
import torchaudio

TEXTO_PRUEBA = 'Hola, esta es una prueba de clonación de voz. ¿Suena parecido a mí?'

for persona, pth_path in archivos_generados.items():
    print(f'\n Probando voz de: {persona}')

    data = torch.load(pth_path, map_location=device)

    out = modelo_xtts.inference(
        text=TEXTO_PRUEBA,
        language='es',
        gpt_cond_latent=data['gpt_cond_latent'].to(device),
        speaker_embedding=data['speaker_embedding'].to(device),
        temperature=0.7,
    )

    test_path = f'/content/test_{persona}.wav'
    torchaudio.save(test_path, torch.tensor(out['wav']).unsqueeze(0), 24000)
    display(Audio(test_path, autoplay=False))


 Probando voz de: BNE_Voice



 Probando voz de: MTC_Voice



 Probando voz de: SLM_Voice


# 6. Exportación de Pesos y Persistencia en Repositorio Remoto

Implementación de la fase de **extracción de artefactos**, diseñada para transferir los embeddings generados desde el entorno efímero de Google Colab hacia el almacenamiento local del usuario y, posteriormente, al repositorio de control de versiones. Este proceso cierra el ciclo de preparación de datos, permitiendo que las "huellas vocales" estén disponibles de forma permanente para aplicaciones de producción.

* **Descarga Síncrona de Pesos (.pth):** Utiliza la librería `google.colab.files` para disparar eventos de descarga directos en el navegador. Al iterar sobre el diccionario de archivos generados, se asegura que cada identidad vocal procesada se guarde localmente con su nombre correspondiente y metadatos integrados.
* **Monitoreo de Carga y Tamaño:** Calcula dinámicamente el peso de cada archivo en KB antes de la transferencia. Este registro de auditoría permite verificar que la serialización de los tensores de PyTorch se realizó correctamente y que no hubo truncamiento de datos durante la escritura en disco.
* **Protocolo de Integración con GitHub:** Establece un flujo de trabajo manual para la actualización del repositorio `DeepLearning`. Al centralizar los archivos `.pth` en la carpeta `cloned_voice/`, se facilita el despliegue de modelos *Fast-Pitch* o *XTTS* en otros entornos (como servidores locales o nubes privadas) sin repetir la fase de extracción.
* **Verificación de Commit y Trazabilidad:** Proporciona una lista de verificación (checklist) visual de los archivos descargados. Esta interfaz de consola sirve como guía de control de calidad para el usuario, garantizando que todos los locutores procesados en la sesión actual sean incluidos en el *commit* final del repositorio.
* **Gestión de Sesión Volátil:** Resuelve el problema de la pérdida de datos en entornos de nube gratuita (donde los archivos locales se eliminan al desconectar el runtime). Al exportar los embeddings como activos binarios, se transforma un proceso costoso de cómputo en un recurso estático reutilizable de bajo peso.

In [25]:
from google.colab import files

print('Descargando embeddings a tu PC...\n')

for persona, pth_path in archivos_generados.items():
    size = os.path.getsize(pth_path) / 1024
    print(f'  Descargando: {persona}_voice.pth ({size:.1f} KB)')
    files.download(pth_path)

print(f'\n{"="*50}')
print(' ¡Descarga completa!')
print()
print(' SIGUIENTE PASO:')
print('   1. Ve a: https://github.com/BrajanNieto/DeepLearning')
print('   2. Abre la carpeta: cloned_voice/')
print('   3. Click "Add file" → "Upload files"')
print('   4. Sube todos los archivos .pth descargados')
print('   5. Haz commit')
print()
print(' Archivos a subir:')
for persona in archivos_generados:
    print(f'   {persona}_voice.pth')
print(f'{"="*50}')

Descargando embeddings a tu PC...

  Descargando: BNE_Voice_voice.pth (132.0 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Descargando: MTC_Voice_voice.pth (132.0 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Descargando: SLM_Voice_voice.pth (132.0 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 ¡Descarga completa!

 SIGUIENTE PASO:
   1. Ve a: https://github.com/BrajanNieto/DeepLearning
   2. Abre la carpeta: cloned_voice/
   3. Click "Add file" → "Upload files"
   4. Sube todos los archivos .pth descargados
   5. Haz commit

 Archivos a subir:
   BNE_Voice_voice.pth
   MTC_Voice_voice.pth
   SLM_Voice_voice.pth
